In [ ]:
"""
Face Detection Evaluation v2
==============================
Evaluates face detection rates using TWO detectors:
    1. RetinaFace  — high accuracy, CVPR 2020
    2. YuNet       — lightweight, OpenCV official

For each detector, compares:
    - Blurred input images      → baseline detection rate
    - Deblurred model outputs   → post-deblur detection rate
    - Recovery rate             → improvement over baseline

Models evaluated: UNet, ViT

Requirements:
    pip install deepface opencv-contrib-python tqdm

Folder structure expected:
    /content/Final_Modeling_Dataset/test/blur/
        1_low.jpg, 1_medium.jpg, 1_high.jpg, 1_very_high.jpg ...

    /content/deblurred_outputs_unet/output_images/
        deblur_0_high.jpg, deblur_0_low.jpg ...

    /content/deblurred_outputs_vit/deblurred_outputs_vit/
        0_high.jpg, 0_low.jpg ...

Outputs saved to:
    /content/drive/MyDrive/detection_evaluation_v2/
        retinaface_results.csv
        yunet_results.csv
        combined_summary.csv
        full_report.txt
"""

'\nFace Detection Evaluation v2\n==============================\nEvaluates face detection rates using TWO detectors:\n    1. RetinaFace  — high accuracy, CVPR 2020\n    2. YuNet       — lightweight, OpenCV official\n\nFor each detector, compares:\n    - Blurred input images      → baseline detection rate\n    - Deblurred model outputs   → post-deblur detection rate\n    - Recovery rate             → improvement over baseline\n\nModels evaluated: UNet, ViT\n\nRequirements:\n    pip install deepface opencv-contrib-python tqdm\n\nFolder structure expected:\n    /content/Final_Modeling_Dataset/test/blur/\n        1_low.jpg, 1_medium.jpg, 1_high.jpg, 1_very_high.jpg ...\n\n    /content/deblurred_outputs_unet/output_images/\n        deblur_0_high.jpg, deblur_0_low.jpg ...\n\n    /content/deblurred_outputs_vit/deblurred_outputs_vit/\n        0_high.jpg, 0_low.jpg ...\n\nOutputs saved to:\n    /content/drive/MyDrive/detection_evaluation_v2/\n        retinaface_results.csv\n        yunet_result

In [ ]:
!gdown https://drive.google.com/uc?id=1zgtvkRJb3y9XhaS1uu9QvYvKxZmmmrGx
!gdown https://drive.google.com/uc?id=1fjUJm5OpWcWnpHOQYiH8GXuMcLaShTxe
!gdown https://drive.google.com/uc?id=1PDfH4iwk6tuc9G1aVuXTPzxEo7YJ1fdL
!gdown https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T

Downloading...
From: https://drive.google.com/uc?id=1zgtvkRJb3y9XhaS1uu9QvYvKxZmmmrGx
To: /content/deblurred_outputs_unet.zip
100% 4.91M/4.91M [00:00<00:00, 34.8MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fjUJm5OpWcWnpHOQYiH8GXuMcLaShTxe
To: /content/deblurred_outputs_vit.zip
100% 8.80M/8.80M [00:00<00:00, 49.5MB/s]
Downloading...
From: https://drive.google.com/uc?id=1PDfH4iwk6tuc9G1aVuXTPzxEo7YJ1fdL
To: /content/deblurred_outputs_vit-unet.zip
100% 11.2M/11.2M [00:00<00:00, 34.1MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T
From (redirected): https://drive.google.com/uc?id=1V48xVqhC-D2DQbkvi4AEZrT_3mYWFl5T&confirm=t&uuid=0c4a8841-82cc-4adb-b1b0-02e682f0906b
To: /content/Final_Modeling_Dataset.zip
100% 163M/163M [00:01<00:00, 83.0MB/s]


In [ ]:
!unzip -o deblurred_outputs_unet.zip -d /content/deblurred_outputs_unet
!unzip -o deblurred_outputs_vit.zip -d /content/deblurred_outputs_vit
!unzip -o deblurred_outputs_vit-unet.zip -d /content/deblurred_outputs_vit-unet
!unzip -o Final_Modeling_Dataset.zip -d /content/Final_Modeling_Dataset

Streaming output truncated to the last 5000 lines.
  inflating: /content/Final_Modeling_Dataset/train/blur/1399_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1907_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1074_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1315_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1211_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/535_very_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/792_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1898_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1890_low.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1608_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1460_medium.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/1225_high.jpg  
  inflating: /content/Final_Modeling_Dataset/train/blur/105_low.jpg  
  inf

In [ ]:
import os
import cv2
import csv
import time
import numpy as np
from tqdm import tqdm

In [ ]:
TEST_BLUR_DIR = "/content/Final_Modeling_Dataset/test/blur"

MODEL_OUTPUT_DIRS = {
    "UNet": {
        "path"  : "/content/deblurred_outputs_unet/output_images",
        "prefix": "deblur_"   # UNet saves as deblur_0_high.jpg
    },
    "ViT": {
        "path"  : "/content/deblurred_outputs_vit/deblurred_outputs_vit",
        "prefix": ""           # ViT saves as 0_high.jpg
    },
    "ViT-UNet": {
        "path"  : "/content/deblurred_outputs_vit-unet/deblurred_outputs",
        "prefix": ""           # Vit_UNet saves as 0_high.jpg
    },
}

BLUR_LEVELS   = ["low", "medium", "high", "very_high"]
SUPPORTED_EXT = (".jpg", ".jpeg", ".png")
RESULTS_DIR   = "/content/drive/MyDrive/detection_evaluation_v2"
YUNET_MODEL   = "/content/face_detection_yunet_2023mar.onnx"

os.makedirs(RESULTS_DIR, exist_ok=True)

In [ ]:
def download_yunet():
    """Download YuNet ONNX model from OpenCV zoo."""
    if os.path.exists(YUNET_MODEL):
        print("YuNet model already downloaded.")
        return
    print("Downloading YuNet model...")
    import urllib.request
    url = "https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx"
    urllib.request.urlretrieve(url, YUNET_MODEL)
    print(f"YuNet model saved to: {YUNET_MODEL}")

In [ ]:
class RetinaFaceDetector:
    """
    RetinaFace detector via DeepFace.
    High accuracy — CVPR 2020.
    """
    def __init__(self):
        from deepface import DeepFace
        self.DeepFace = DeepFace
        print("RetinaFace detector initialized.")

    def detect(self, image_path):
        """Returns True if at least one face detected."""
        try:
            result = self.DeepFace.extract_faces(
                img_path         = image_path,
                detector_backend = "retinaface",
                enforce_detection= False,
                align             = False,
            )
            for face in result:
                if face.get("confidence", 0) > 0.9:
                    return True
            return False
        except Exception:
            return False

In [ ]:
class YuNetDetector:
    """
    YuNet detector via OpenCV.
    Lightweight — OpenCV official model.
    """
    def __init__(self, model_path, conf_threshold=0.9):
        self.detector = cv2.FaceDetectorYN.create(
            model_path,
            "",
            (320, 320),
            conf_threshold,
            0.3,
            5000
        )
        print("YuNet detector initialized.")

    def detect(self, image_path):
        """Returns True if at least one face detected."""
        img = cv2.imread(image_path)
        if img is None:
            return False
        h, w = img.shape[:2]
        self.detector.setInputSize((w, h))
        _, faces = self.detector.detect(img)
        return faces is not None and len(faces) > 0

In [ ]:
def get_files_by_level(directory, level):
    """
    Return sorted list of image files in directory that exactly
    match the given blur level suffix.
    Uses split-based matching to correctly distinguish
    'high' from 'very_high' without fragile endswith guards.
    """
    result = []
    for f in os.listdir(directory):
        if not f.lower().endswith(SUPPORTED_EXT):
            continue
        stem = os.path.splitext(f)[0]
        # exact suffix match — prevents _very_high matching _high
        if stem.endswith(f"_{level}") and not stem.endswith(f"_very_{level}"):
            result.append(f)
    return sorted(result)


def get_deblurred_filename(blurred_filename, prefix):
    """Map a blurred filename to its expected deblurred counterpart."""
    return f"{prefix}{blurred_filename}"

In [ ]:
def run_evaluation(detector, detector_name):
    print(f"\n{'='*55}")
    print(f"EVALUATING WITH: {detector_name}")
    print(f"{'='*55}")

    # ── STEP 1: Baseline on blurred inputs ──
    print(f"\n[{detector_name}] Step 1 — Baseline detection on blurred inputs")
    baseline = {}

    for level in BLUR_LEVELS:
        files    = get_files_by_level(TEST_BLUR_DIR, level)
        detected = 0
        total    = len(files)

        for filename in tqdm(files, desc=f"  Baseline [{level:<10}]"):
            if detector.detect(os.path.join(TEST_BLUR_DIR, filename)):
                detected += 1

        pct = 100 * detected / total if total > 0 else 0
        baseline[level] = {"detected": detected, "total": total, "pct": pct}
        print(f"  {level:<12}: {detected}/{total} ({pct:.1f}%)")

    base_total_det = sum(v["detected"] for v in baseline.values())
    base_total_n   = sum(v["total"]    for v in baseline.values())
    base_overall   = 100 * base_total_det / base_total_n if base_total_n > 0 else 0
    print(f"\n  Overall baseline: {base_total_det}/{base_total_n} ({base_overall:.1f}%)")

    # ── STEP 2: Detection on deblurred outputs ──
    print(f"\n[{detector_name}] Step 2 — Detection on deblurred model outputs")
    model_results = {}

    for model_name, model_config in MODEL_OUTPUT_DIRS.items():
        print(f"\n  [{model_name}]")
        model_results[model_name] = {}
        output_dir = model_config["path"]
        prefix     = model_config["prefix"]

        if not os.path.exists(output_dir):
            print(f"  WARNING: Directory not found — {output_dir}")
            for level in BLUR_LEVELS:
                model_results[model_name][level] = {
                    "detected": 0, "total": 0, "pct": 0}
            continue

        for level in BLUR_LEVELS:
            blurred_files = get_files_by_level(TEST_BLUR_DIR, level)
            detected = total = missing = 0

            for blur_file in tqdm(blurred_files,
                                  desc=f"    {model_name} [{level:<10}]"):
                deblurred_file = get_deblurred_filename(blur_file, prefix)
                deblurred_path = os.path.join(output_dir, deblurred_file)

                if not os.path.exists(deblurred_path):
                    missing += 1
                    continue

                total += 1
                if detector.detect(deblurred_path):
                    detected += 1

            pct      = 100 * detected / total if total > 0 else 0
            recovery = pct - baseline[level]["pct"]
            sign     = "+" if recovery >= 0 else ""
            model_results[model_name][level] = {
                "detected": detected, "total": total, "pct": pct}

            if missing > 0:
                print(f"  WARNING: {missing} files missing for {model_name} [{level}]")
            print(f"    {level:<12}: {detected}/{total} ({pct:.1f}%) | "
                  f"Recovery: {sign}{recovery:.1f}% | Missing: {missing}")

    return baseline, base_overall, model_results

In [ ]:
def print_results_table(detector_name, baseline, base_overall, model_results):
    print(f"\n{'='*70}")
    print(f"RESULTS TABLE — {detector_name}")
    print(f"{'='*70}")

    # Detection rate table
    print(f"\nDetection Rate (%):")
    print(f"  {'Model':<14} {'Low':>8} {'Medium':>8} {'High':>8} {'V.High':>8} {'Overall':>8}")
    print("  " + "-"*56)

    base_vals = [f"{baseline[l]['pct']:.1f}%" for l in BLUR_LEVELS]
    print(f"  {'Blurred':<14} "
          f"{base_vals[0]:>8} {base_vals[1]:>8} "
          f"{base_vals[2]:>8} {base_vals[3]:>8} {base_overall:>7.1f}%")

    for model_name in MODEL_OUTPUT_DIRS:
        vals = []
        total_d = total_n = 0
        for l in BLUR_LEVELS:
            d = model_results[model_name][l]["detected"]
            n = model_results[model_name][l]["total"]
            total_d += d
            total_n += n
            vals.append(f"{100*d/n:.1f}%" if n > 0 else "N/A")
        overall = 100 * total_d / total_n if total_n > 0 else 0
        print(f"  {model_name:<14} "
              f"{vals[0]:>8} {vals[1]:>8} "
              f"{vals[2]:>8} {vals[3]:>8} {overall:>7.1f}%")

    # Recovery rate table
    print(f"\nDetection Recovery Rate (improvement over blurred baseline %):")
    print(f"  {'Model':<14} {'Low':>8} {'Medium':>8} {'High':>8} {'V.High':>8} {'Overall':>8}")
    print("  " + "-"*56)

    for model_name in MODEL_OUTPUT_DIRS:
        vals = []
        total_d = total_n = 0
        for l in BLUR_LEVELS:
            d       = model_results[model_name][l]["detected"]
            n       = model_results[model_name][l]["total"]
            total_d += d
            total_n += n
            pct     = 100 * d / n if n > 0 else 0
            delta   = pct - baseline[l]["pct"]
            sign    = "+" if delta >= 0 else ""
            vals.append(f"{sign}{delta:.1f}%")
        overall  = 100 * total_d / total_n if total_n > 0 else 0
        recovery = overall - base_overall
        sign     = "+" if recovery >= 0 else ""
        print(f"  {model_name:<14} "
              f"{vals[0]:>8} {vals[1]:>8} "
              f"{vals[2]:>8} {vals[3]:>8} {sign}{recovery:>7.1f}%")

In [ ]:
def save_results_csv(detector_name, baseline, base_overall, model_results):
    filename = f"{detector_name.lower().replace(' ', '_')}_results.csv"
    csv_path = os.path.join(RESULTS_DIR, filename)

    with open(csv_path, "w", newline="") as f:
        fieldnames = ["model", "blur_level", "detected",
                      "total", "detection_rate_pct", "recovery_pct"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for level in BLUR_LEVELS:
            writer.writerow({
                "model"             : "Blurred_Baseline",
                "blur_level"        : level,
                "detected"          : baseline[level]["detected"],
                "total"             : baseline[level]["total"],
                "detection_rate_pct": round(baseline[level]["pct"], 2),
                "recovery_pct"      : 0.0,
            })

        for model_name in MODEL_OUTPUT_DIRS:
            for level in BLUR_LEVELS:
                d        = model_results[model_name][level]["detected"]
                n        = model_results[model_name][level]["total"]
                pct      = 100 * d / n if n > 0 else 0
                recovery = pct - baseline[level]["pct"]
                writer.writerow({
                    "model"             : model_name,
                    "blur_level"        : level,
                    "detected"          : d,
                    "total"             : n,
                    "detection_rate_pct": round(pct, 2),
                    "recovery_pct"      : round(recovery, 2),
                })

    print(f"\n  CSV saved: {csv_path}")
    return csv_path


def save_combined_summary(all_results):
    """Save combined_summary.csv merging results from all detectors."""
    csv_path = os.path.join(RESULTS_DIR, "combined_summary.csv")

    with open(csv_path, "w", newline="") as f:
        fieldnames = ["detector", "model", "blur_level",
                      "detected", "total", "detection_rate_pct", "recovery_pct"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for detector_name, (baseline, base_overall, model_results) in all_results.items():
            for level in BLUR_LEVELS:
                writer.writerow({
                    "detector"          : detector_name,
                    "model"             : "Blurred_Baseline",
                    "blur_level"        : level,
                    "detected"          : baseline[level]["detected"],
                    "total"             : baseline[level]["total"],
                    "detection_rate_pct": round(baseline[level]["pct"], 2),
                    "recovery_pct"      : 0.0,
                })
            for model_name in MODEL_OUTPUT_DIRS:
                for level in BLUR_LEVELS:
                    d        = model_results[model_name][level]["detected"]
                    n        = model_results[model_name][level]["total"]
                    pct      = 100 * d / n if n > 0 else 0
                    recovery = pct - baseline[level]["pct"]
                    writer.writerow({
                        "detector"          : detector_name,
                        "model"             : model_name,
                        "blur_level"        : level,
                        "detected"          : d,
                        "total"             : n,
                        "detection_rate_pct": round(pct, 2),
                        "recovery_pct"      : round(recovery, 2),
                    })

    print(f"  Combined summary saved: {csv_path}")
    return csv_path

In [ ]:
def save_full_report(all_results):
    report_path = os.path.join(RESULTS_DIR, "full_report.txt")

    with open(report_path, "w") as f:
        f.write("FACE DETECTION EVALUATION REPORT\n")
        f.write("Detectors: RetinaFace, YuNet\n")
        f.write("="*70 + "\n\n")

        for detector_name, (baseline, base_overall, model_results) in all_results.items():
            f.write(f"DETECTOR: {detector_name}\n")
            f.write("-"*50 + "\n")
            f.write(f"{'Model':<14} {'Low':>8} {'Medium':>8} "
                    f"{'High':>8} {'V.High':>8} {'Overall':>8} {'Recovery':>10}\n")

            base_vals = " ".join([f"{baseline[l]['pct']:>7.1f}%" for l in BLUR_LEVELS])
            f.write(f"{'Blurred':<14} {base_vals} {base_overall:>7.1f}%"
                    f"  {'—':>10}\n")

            for model_name in MODEL_OUTPUT_DIRS:
                total_d = total_n = 0
                vals = []
                for l in BLUR_LEVELS:
                    d = model_results[model_name][l]["detected"]
                    n = model_results[model_name][l]["total"]
                    total_d += d
                    total_n += n
                    vals.append(f"{100*d/n:>7.1f}%" if n > 0 else "    N/A")
                overall  = 100 * total_d / total_n if total_n > 0 else 0
                recovery = overall - base_overall
                sign     = "+" if recovery >= 0 else ""
                f.write(f"{model_name:<14} {' '.join(vals)} "
                        f"{overall:>7.1f}%  {sign}{recovery:.1f}%\n")
            f.write("\n")

    print(f"  Full report saved: {report_path}")

In [ ]:
def run_face_detection_evaluation():
    print("Installing requirements...")
    os.system("pip install -q deepface opencv-contrib-python")

    download_yunet()

    print("\nInitializing detectors...")
    retinaface = RetinaFaceDetector()
    yunet      = YuNetDetector(YUNET_MODEL)

    detectors = {
        "RetinaFace": retinaface,
        "YuNet"     : yunet,
    }

    all_results = {}

    for detector_name, detector in detectors.items():
        baseline, base_overall, model_results = run_evaluation(
            detector, detector_name)
        print_results_table(
            detector_name, baseline, base_overall, model_results)
        save_results_csv(
            detector_name, baseline, base_overall, model_results)
        all_results[detector_name] = (baseline, base_overall, model_results)

    save_combined_summary(all_results)
    save_full_report(all_results)

    print("\n" + "="*55)
    print("EVALUATION COMPLETE")
    print("="*55)
    print(f"Results saved to: {RESULTS_DIR}")
    print("\nFiles generated:")
    for f in sorted(os.listdir(RESULTS_DIR)):
        print(f"  {f}")


run_face_detection_evaluation()

Installing requirements...
YuNet model already downloaded.

Initializing detectors...
RetinaFace detector initialized.
YuNet detector initialized.

EVALUATING WITH: RetinaFace

[RetinaFace] Step 1 — Baseline detection on blurred inputs


  Baseline [low       ]: 100%|██████████| 156/156 [00:20<00:00,  7.58it/s]


  low         : 156/156 (100.0%)


  Baseline [medium    ]: 100%|██████████| 156/156 [00:20<00:00,  7.73it/s]


  medium      : 156/156 (100.0%)


  Baseline [high      ]: 100%|██████████| 156/156 [00:19<00:00,  7.83it/s]


... (55 outputs left out)

    ViT-UNet [high      ]: 100%|██████████| 156/156 [00:01<00:00, 138.87it/s]


    high        : 149/156 (95.5%) | Recovery: +75.0% | Missing: 0


    ViT-UNet [very_high ]: 100%|██████████| 156/156 [00:00<00:00, 161.41it/s]

    very_high   : 149/156 (95.5%) | Recovery: +94.2% | Missing: 0

RESULTS TABLE — YuNet

Detection Rate (%):
  Model               Low   Medium     High   V.High  Overall
  --------------------------------------------------------
  Blurred           96.8%    85.3%    20.5%     1.3%    51.0%
  UNet              98.7%    98.1%    97.4%    94.9%    97.3%
  ViT               96.2%    91.0%    78.8%    59.0%    81.2%
  ViT-UNet          97.4%    97.4%    95.5%    95.5%    96.5%

Detection Recovery Rate (improvement over blurred baseline %):
  Model               Low   Medium     High   V.High  Overall
  --------------------------------------------------------
  UNet              +1.9%   +12.8%   +76.9%   +93.6% +   46.3%
  ViT               -0.6%    +5.8%   +58.3%   +57.7% +   30.3%
  ViT-UNet          +0.6%   +12.2%   +75.0%   +94.2% +   45.5%

  CSV saved: /content/drive/MyDrive/detection_evaluation_v2/yunet_results.csv
  Combined summary saved: /content/drive/MyDrive/detection_evaluatio